# Chat Agent

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction LR
INIT --> CHAT
CHAT --> FINAL
```


### a) Create Agent

In [1]:
import os
os.environ["LOG_LEVEL"]="WARNING"

In [2]:
from gai.asm.agents import ChatAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper

from gai.lib.tests import make_local_tmp
import os
here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
from gai.messages import FileMonologue
monologue = FileMonologue(agent_name="ChatAgent",file_path=file_path)

# Create an artificial dialogue history for testing
from gai.messages import FileDialogue, MessagePydantic
messages = [
    MessagePydantic(**{
        'id': 'b1e5f98c-f6eb-47de-a6e2-387510d970f9', 
        'header': {
            'sender': 'User',
            'recipient': 'Sara',
            "timestamp": 1751308157.270983,
            "order": 0            
        }, 'body': {
            'type': 'chat.send',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0,
            'step_no': 0,
            'role': "user",
            'content': 'I love horror stories, are you familiar with them?',
        }
    }),
    MessagePydantic(**{
        'id': 'abbc7961-45dc-4973-aaf4-a6224ed35d37', 
        'header': {
            'sender': 'Sara',
            'recipient': 'User',
            "timestamp": 1751308167.3488164,
            "order": 1
        }, 'body': {
            'type': 'chat.reply',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0, 
            'step_no': 1,
            'chunk_no':10,
            'chunk':'<eom>',
            'role': "assistant",
            'content': 'Yes, I am familiar with horror stories. They are a fascinating genre that can evoke strong emotions and create a sense of suspense and fear. Do you have any specific horror stories in mind that you would like to discuss?'
        }
    })]
from gai.lib.constants import DEFAULT_GUID
file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(messages=messages,file_path=file_path)
recap = dialogue.extract_recap()

aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ChatAgent(
    agent_name="Sara",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    monologue=monologue
)

### b) run and infer the context from dialogue

In [3]:
monologue.reset()

user_message="Tell me a one paragraph story."

resp=agent.run(user_message=user_message, recap=recap)
# Stream the response
async for chunk in resp:
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)
            
# Update dialogue
dialogue.add_user_message(recipient="Sara", content=user_message)
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)

Hi there! I'm Sara. Here's a horror story for you: The old music box had been silent for decades, collecting dust in Margaret's grandmother's attic until she wound the tiny brass key one rainy evening. As the haunting melody began to play, the porcelain ballerina inside started her delicate pirouettes, but Margaret noticed something chilling—with each turn, the dancer's painted smile seemed to grow wider, more knowing. The music slowed to an unnatural crawl, yet the ballerina continued spinning faster and faster, her tiny arms reaching outward as if beckoning. When Margaret tried to close the lid, it wouldn't budge, and she realized with growing terror that the melody wasn't coming from the music box anymore—it was coming from behind her, hummed in a voice that sounded exactly like her own.

MessagePydantic(id='e18bd0e9-dcd3-41fd-884f-4a6ced67631a', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1753271787.1493258, order=3), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=1, step_no=1, message_id='00000000-0000-0000-0000-000000000000.16', chunk_no=0, chunk='<eom>', content_type='text', role='assistant', content="Hi there! I'm Sara. Here's a horror story for you: The old music box had been silent for decades, collecting dust in Margaret's grandmother's attic until she wound the tiny brass key one rainy evening. As the haunting melody began to play, the porcelain ballerina inside started her delicate pirouettes, but Margaret noticed something chilling—with each turn, the dancer's painted smile seemed to grow wider, more knowing. The music slowed to an unnatural crawl, yet the ballerina continued spinning faster and faster, her tiny arms reaching outward as if beckoning. When Margaret tried to

### c) Show monologue

In [4]:
import json

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.fsm.monologue.list_messages()
for message in messages[-2:]:
    print(json.dumps(message.model_dump(), indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size = agent.fsm.monologue.get_total_size()
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "id": "ee423a19-00c0-4eb4-ae42-27ed438910bf",
    "header": {
        "sender": "User",
        "recipient": "ChatAgent",
        "timestamp": 1753271773.822109,
        "order": 0
    },
    "body": {
        "type": "monologue",
        "state_name": "CHAT",
        "step_no": 1,
        "content_type": "text",
        "role": "user",
        "content": "\n            Your name is Sara within the context of this conversation and you will always respond as such.\n            Do not refer to yourself as an AI or a bot or confuse your name with other agents.\n           \n            You may respond to my following message using the context you have learnt.\n            \n            Tell me a one paragraph story.\n\n            Here is a recap of the conversation:\n            User: I love horror stories, are you familiar with them?\nSara: Yes, I am familiar with horror stories. They are a fascinating genre that

### d) Show dialogue

In [5]:
for msg in dialogue.list_messages():
    print(f"{msg.header.sender}: {msg.body.content}")

User: I love horror stories, are you familiar with them?
Sara: Yes, I am familiar with horror stories. They are a fascinating genre that can evoke strong emotions and create a sense of suspense and fear. Do you have any specific horror stories in mind that you would like to discuss?
User: Sara, Tell me a one paragraph story.
Sara: Hi there! I'm Sara. Here's a horror story for you: The old music box had been silent for decades, collecting dust in Margaret's grandmother's attic until she wound the tiny brass key one rainy evening. As the haunting melody began to play, the porcelain ballerina inside started her delicate pirouettes, but Margaret noticed something chilling—with each turn, the dancer's painted smile seemed to grow wider, more knowing. The music slowed to an unnatural crawl, yet the ballerina continued spinning faster and faster, her tiny arms reaching outward as if beckoning. When Margaret tried to close the lid, it wouldn't budge, and she realized with growing terror that t